# Проверка карточек моделей

Компактная проверка версий моделей, находящихся в эксплуатации.

**Флаги:** `1` — обнаружен дефект, `0` — дефект не обнаружен.

Логика согласована с `Schema.txt` и предоставленным сэмплом. Физическая доступность файлов, репозиториев и таблиц не проверяется.

## 1. Импорты и настройки окружения

In [ ]:
import os
import re
import sys
from datetime import date, datetime
from functools import reduce
from pathlib import Path

# Spark-клиент из рабочего окружения. Блок безопасно пропускается вне кластера.
SPARK_CLIENT = Path("/usr/sdp/current/spark3.5.1-client")
if SPARK_CLIENT.exists():
    os.environ.setdefault("SPARK_MAJOR_VERSION", "3.5.1")
    os.environ.setdefault("SPARK_HOME", str(SPARK_CLIENT))
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    sys.path.insert(0, str(SPARK_CLIENT / "python"))
    sys.path.insert(0, str(SPARK_CLIENT / "python/lib/py4j-0.10.9.7-src.zip"))

from pyspark import SparkConf
from pyspark.sql import SparkSession, Window, functions as F
from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter


SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
MAX_EXCEL_ROWS = 1_048_575

# Для обычной проверки заполненности. Не применимо / $not_applicable$ здесь
# считаются заполненными значениями: это отдельное бизнес-состояние, а не NULL.
EMPTY_TEXT = {"", "null", "none", "nan", "-", "[]", "{}"}
NO_RESULT_TEXT = EMPTY_TEXT | {"отсутствует", "$not_applicable$", "не применимо"}

# Поле SAMPLE_DATA_ADDTNL_INFO_TXT по схеме содержит путь к выборке и/или
# дополнительную информацию. Для флага данных засчитываем только явный путь,
# URL или имя таблицы, а не произвольный комментарий.
LOCATION_PATTERN = (
    r"(?i)(?:"
    r"(?:https?|hdfs|s3a?|abfss?|wasbs?|file|dbfs)://[^\s<>]+"
    r"|\\\\[^\s\\]+\\[^\s]+"
    r"|(?:^|\s)/(?:[^\s/]+/)+[^\s]*"
    r"|(?:^|\s)[a-z]:\\[^\s]+"
    r"|\b[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*(?:\.[a-z_][a-z0-9_]*)?\b"
    r")"
)

## 2. Spark-сессия

In [ ]:
spark = SparkSession.getActiveSession()

if spark is None:
    conf = (
        SparkConf()
        .setAppName("model_card_audit")
        .setMaster("yarn")
        .set("spark.executor.cores", "2")
        .set("spark.executor.memory", "6g")
        .set("spark.executor.memoryOverhead", "1g")
        .set("spark.driver.maxResultSize", "4g")
        .set("spark.shuffle.service.enabled", "true")
        .set("spark.dynamicAllocation.enabled", "true")
        .set("spark.dynamicAllocation.initialExecutors", "4")
        .set("spark.dynamicAllocation.maxExecutors", "12")
        .set("spark.port.maxRetries", "150")
    )
    spark = (
        SparkSession.builder
        .config(conf=conf)
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

## 3. Параметры запуска

In [ ]:
AS_OF = date.today().isoformat()
OUTPUT_FILE = Path.cwd() / "model_card_audit.xlsx"

print("Дата среза:", AS_OF)
print("Файл:", OUTPUT_FILE)

## 4. Вспомогательные функции

In [ ]:
def normalized(col):
    return F.lower(F.trim(F.regexp_replace(col.cast("string"), "\u00a0", " ")))


def filled(col):
    value = normalized(col)
    empty_array = value.rlike(r'^\[\s*(?:null|""|\s|,)*\s*\]$')
    return F.coalesce((~value.isin(*EMPTY_TEXT)) & (~empty_array), F.lit(False))


def result_present(col):
    value = normalized(col)
    return F.coalesce(~value.isin(*NO_RESULT_TEXT), F.lit(False))


def true_value(col):
    return F.coalesce(normalized(col).isin("true", "1", "да"), F.lit(False))


def has_location(col):
    return filled(col) & F.coalesce(col.cast("string").rlike(LOCATION_PATTERN), F.lit(False))


def has_bitbucket(col):
    return filled(col) & F.coalesce(normalized(col).contains("bitbucket"), F.lit(False))


def read_current(spark, table, keys, fields, as_of):
    """Текущий бизнес-срез таблицы, одна запись на ключ.

    Последнее техническое состояние выбирается внутри бизнес-среза.
    Удалённая (CTL_ACTION=D) последняя запись не заменяется старой записью.
    """
    full_name = f"{SOURCE_DB}.{table}"
    if not spark.catalog.tableExists(full_name):
        raise RuntimeError(f"Не найдена таблица: {full_name}")

    raw = spark.table(full_name)
    lower = [c.lower() for c in raw.columns]
    if len(lower) != len(set(lower)):
        raise RuntimeError(f"{full_name}: неоднозначные имена столбцов")
    raw = raw.toDF(*lower)

    required = set(keys + fields + ["start_dt"])
    missing = sorted(required - set(raw.columns))
    if missing:
        raise RuntimeError(f"{full_name}: отсутствуют поля: {', '.join(missing)}")

    meta = [c for c in ("start_dt", "end_dt", "ctl_datechange", "ctl_loading", "ctl_action") if c in raw.columns]
    df = raw.select(*dict.fromkeys(keys + fields + meta))

    # SID приводим к одному типу, чтобы joins не зависели от физического типа.
    for name in df.columns:
        if name.endswith("_sid"):
            df = df.withColumn(name, F.col(name).cast("string"))

    bound = F.lit(as_of).cast("date")
    df = df.filter(F.col("start_dt").cast("date") <= bound)
    if "end_dt" in df.columns:
        end_date = F.col("end_dt").cast("date")
        df = df.filter(end_date.isNull() | (end_date >= bound))

    order = []
    if "ctl_datechange" in df.columns:
        order.append(F.col("ctl_datechange").desc_nulls_last())
    order.append(F.col("start_dt").desc_nulls_last())
    if "ctl_loading" in df.columns:
        order.append(F.col("ctl_loading").desc_nulls_last())

    w = Window.partitionBy(*keys).orderBy(*order)
    df = df.withColumn("__rn", F.row_number().over(w)).filter(F.col("__rn") == 1).drop("__rn")

    if "ctl_action" in df.columns:
        df = df.filter(F.coalesce(F.upper(F.trim(F.col("ctl_action"))) != "D", F.lit(True)))

    return df.select(*dict.fromkeys(keys + fields))


def missing_any(columns):
    return reduce(lambda a, b: a | b, [~filled(F.col(c)) for c in columns], F.lit(False))

## 5. Формирование отчёта

In [ ]:
def build_report(spark, as_of):
    """Строит итоговый DataFrame проверки; ничего не сохраняет на диск."""
    date.fromisoformat(as_of)
    spark.sparkContext.setLogLevel("ERROR")

    models = read_current(
        spark,
        "t_model",
        ["model_sid"],
        ["model_code", "model_name"],
        as_of,
    )

    versions = read_current(
        spark,
        "t_model_ver",
        ["model_ver_sid"],
        [
            "model_sid",
            "model_ver_num",
            "model_ver_name",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name",
            "model_ver_owner_block_name",
            "model_ver_owner_dprtmt_name",
            "model_ver_owner_cont_name",
            "model_ver_dev_block_name",
            "model_ver_dev_dprtmt_name",
            "model_ver_dev_dprtmt_cont_name",
            "model_ver_dev_start_fact_dttm",
            "model_ver_dev_end_fact_dttm",
            "model_ver_dev_report_sid",
            "model_ver_repstry_link_txt",
            "model_ver_mlstorage_branch_link_txt",
            "model_ver_montrg_last_rslt_name",
            "model_ver_montrg_last_rslt_dttm",
            "model_ver_montrg_last_sid",
            "model_ver_montrg_auto_exclude_reason_array",
        ],
        as_of,
    )

    analytics = read_current(
        spark,
        "t_model_ver_anlt_dtl",
        ["model_ver_sid"],
        [
            "model_ver_prom_expl_flag",
            "model_ver_not_valid_flag",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name",
            "model_ver_busn_proc_code",
            "model_ver_prom_actual_implm_first_dttm",
            "model_ver_prom_implm_first_dttm",
        ],
        as_of,
    ).select(
        "model_ver_sid",
        "model_ver_prom_expl_flag",
        "model_ver_not_valid_flag",
        F.col("model_ver_signfcnt_ctgry_code").alias("anlt_significance_code"),
        F.col("model_ver_signfcnt_lvl_name").alias("anlt_significance_name"),
        "model_ver_busn_proc_code",
        "model_ver_prom_actual_implm_first_dttm",
        "model_ver_prom_implm_first_dttm",
    )

    # Только версии, реально находящиеся в эксплуатации по аналитической витрине.
    base = (
        versions.join(analytics, "model_ver_sid", "inner")
        .filter(true_value(F.col("model_ver_prom_expl_flag")))
        .join(models, "model_sid", "left")
        .withColumn(
            "expl_start_dttm",
            F.coalesce(
                F.col("model_ver_prom_actual_implm_first_dttm"),
                F.col("model_ver_prom_implm_first_dttm"),
            ),
        )
        .withColumn(
            "significance",
            F.coalesce(F.col("model_ver_signfcnt_ctgry_code"), F.col("anlt_significance_code")),
        )
        .withColumn(
            "significance_name",
            F.coalesce(F.col("model_ver_signfcnt_lvl_name"), F.col("anlt_significance_name")),
        )
    )

    # Уникальность кода модели проверяем без учёта регистра и пробелов по краям.
    duplicate_codes = (
        models.filter(filled(F.col("model_code")))
        .withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .groupBy("__code")
        .agg(F.countDistinct("model_sid").alias("__code_count"))
    )
    base = (
        base.withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .join(duplicate_codes, "__code", "left")
    )

    # -------------------------------------------------------------------------
    # Validation: наличие результата и явное исключение "валидация не требуется"
    # -------------------------------------------------------------------------
    validations = read_current(
        spark,
        "t_valid",
        ["valid_sid"],
        ["model_ver_sid", "valid_stts_name", "valid_rslt_name"],
        as_of,
    )

    validation_stats = validations.groupBy("model_ver_sid").agg(
        F.max(result_present(F.col("valid_rslt_name")).cast("int")).alias("has_validation_result"),
        F.max((normalized(F.col("valid_stts_name")) == "validation_finance_not_needed").cast("int")).alias(
            "validation_not_required_by_status"
        ),
    )
    base = base.join(validation_stats, "model_ver_sid", "left").fillna(
        {"has_validation_result": 0, "validation_not_required_by_status": 0}
    )
    base = base.withColumn(
        "validation_not_required",
        (
            true_value(F.col("model_ver_not_valid_flag"))
            | (F.col("validation_not_required_by_status") == 1)
        ).cast("int"),
    )

    # -------------------------------------------------------------------------
    # Monitoring: карточка версии + родительская сущность мониторинга.
    # Не используем *_RSLT_END_DTTM как условие существования результата:
    # по сэмплу валидный LAST_RSLT может быть у MONITORING_FINANCE_DEVELOPMENT.
    # -------------------------------------------------------------------------
    auto_mon = read_current(
        spark,
        "t_montrg_auto",
        ["montrg_auto_sid"],
        ["montrg_auto_rslt_last_name", "montrg_auto_rslt_last_dttm"],
        as_of,
    ).select(
        F.col("montrg_auto_sid").alias("__auto_sid"),
        F.col("montrg_auto_rslt_last_name").alias("__auto_result"),
        F.col("montrg_auto_rslt_last_dttm").alias("__auto_result_dttm"),
    )

    manual_mon = read_current(
        spark,
        "t_montrg_manual",
        ["montrg_manual_sid"],
        ["montrg_manual_last_rslt_name", "montrg_manual_last_rslt_dttm"],
        as_of,
    ).select(
        F.col("montrg_manual_sid").alias("__manual_sid"),
        F.col("montrg_manual_last_rslt_name").alias("__manual_result"),
        F.col("montrg_manual_last_rslt_dttm").alias("__manual_result_dttm"),
    )

    base = (
        base.join(auto_mon, F.col("model_ver_montrg_last_sid") == F.col("__auto_sid"), "left")
        .join(manual_mon, F.col("model_ver_montrg_last_sid") == F.col("__manual_sid"), "left")
        .withColumn("entity_monitor_result", F.coalesce(F.col("__auto_result"), F.col("__manual_result")))
        .withColumn(
            "monitor_result_available",
            (
                result_present(F.col("model_ver_montrg_last_rslt_name"))
                | result_present(F.col("entity_monitor_result"))
            ).cast("int"),
        )
        .withColumn(
            "monitor_result_mismatch",
            (
                result_present(F.col("model_ver_montrg_last_rslt_name"))
                & result_present(F.col("entity_monitor_result"))
                & (normalized(F.col("model_ver_montrg_last_rslt_name")) != normalized(F.col("entity_monitor_result")))
            ).cast("int"),
        )
    )

    # -------------------------------------------------------------------------
    # Samples / data paths.
    # -------------------------------------------------------------------------
    samples = read_current(
        spark,
        "t_sample_data",
        ["sample_data_sid"],
        [
            "model_ver_sid",
            "valid_sid",
            "montrg_manual_rslt_sid",
            "montrg_auto_rslt_sid",
            "sample_data_type_name",
            "sample_data_addtnl_info_txt",
        ],
        as_of,
    )

    # Валидационные данные: прямая связь VALID_SID или версия + тип "Для валидации".
    valid_sample_direct = (
        samples.filter(filled(F.col("valid_sid")) & has_location(F.col("sample_data_addtnl_info_txt")))
        .join(validations.select("valid_sid", "model_ver_sid"), "valid_sid", "inner")
        .groupBy("model_ver_sid")
        .agg(F.lit(1).alias("__valid_data_direct"))
    )
    valid_sample_by_type = (
        samples.filter(
            filled(F.col("model_ver_sid"))
            & normalized(F.col("sample_data_type_name")).contains("валидац")
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .groupBy("model_ver_sid")
        .agg(F.lit(1).alias("__valid_data_type"))
    )
    base = (
        base.join(valid_sample_direct, "model_ver_sid", "left")
        .join(valid_sample_by_type, "model_ver_sid", "left")
        .withColumn(
            "has_validation_data_path",
            ((F.coalesce(F.col("__valid_data_direct"), F.lit(0)) == 1) | (F.coalesce(F.col("__valid_data_type"), F.lit(0)) == 1)).cast("int"),
        )
    )

    # Для мониторинга связываем SAMPLE_DATA с результатами мониторинга через *_RSLT_SID.
    auto_result_map = read_current(
        spark,
        "t_montrg_auto_rslt",
        ["montrg_auto_rslt_sid"],
        ["montrg_auto_sid"],
        as_of,
    )
    manual_result_map = read_current(
        spark,
        "t_montrg_manual_rslt",
        ["montrg_manual_rslt_sid"],
        ["montrg_manual_sid"],
        as_of,
    )

    auto_monitor_data = (
        samples.filter(filled(F.col("montrg_auto_rslt_sid")) & has_location(F.col("sample_data_addtnl_info_txt")))
        .join(auto_result_map, "montrg_auto_rslt_sid", "inner")
        .groupBy("montrg_auto_sid")
        .agg(F.lit(1).alias("__auto_data"))
    )
    manual_monitor_data = (
        samples.filter(filled(F.col("montrg_manual_rslt_sid")) & has_location(F.col("sample_data_addtnl_info_txt")))
        .join(manual_result_map, "montrg_manual_rslt_sid", "inner")
        .groupBy("montrg_manual_sid")
        .agg(F.lit(1).alias("__manual_data"))
    )
    version_monitor_data = (
        samples.filter(
            filled(F.col("model_ver_sid"))
            & normalized(F.col("sample_data_type_name")).rlike(r"монитор|стабил")
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .groupBy("model_ver_sid")
        .agg(F.lit(1).alias("__version_monitor_data"))
    )

    base = (
        base.join(auto_monitor_data, F.col("model_ver_montrg_last_sid") == F.col("montrg_auto_sid"), "left")
        .join(manual_monitor_data, F.col("model_ver_montrg_last_sid") == F.col("montrg_manual_sid"), "left")
        .join(version_monitor_data, "model_ver_sid", "left")
        .withColumn(
            "has_monitor_data_path",
            (
                (F.coalesce(F.col("__auto_data"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__manual_data"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__version_monitor_data"), F.lit(0)) == 1)
            ).cast("int"),
        )
    )

    # -------------------------------------------------------------------------
    # Binary flags: 1 = defect, 0 = no defect.
    # -------------------------------------------------------------------------
    base = (
        base
        .withColumn("F01_MODEL_CODE", (~filled(F.col("model_code"))).cast("int"))
        .withColumn("F02_MODEL_NAME", (~filled(F.col("model_name"))).cast("int"))
        .withColumn("F03_MODEL_CODE_DUP", (F.coalesce(F.col("__code_count"), F.lit(0)) > 1).cast("int"))
        .withColumn("F04_VERSION_NUM", (~filled(F.col("model_ver_num"))).cast("int"))
        .withColumn("F05_VERSION_NAME", (~filled(F.col("model_ver_name"))).cast("int"))
        .withColumn("F06_DEV_START", (~filled(F.col("model_ver_dev_start_fact_dttm"))).cast("int"))
        .withColumn("F07_DEV_END", (~filled(F.col("model_ver_dev_end_fact_dttm"))).cast("int"))
        .withColumn("F08_EXPL_START", (~filled(F.col("expl_start_dttm"))).cast("int"))
        .withColumn(
            "F09_OWNER",
            missing_any(["model_ver_owner_block_name", "model_ver_owner_dprtmt_name", "model_ver_owner_cont_name"]).cast("int"),
        )
        .withColumn(
            "F10_DEVELOPER",
            missing_any(["model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_dev_dprtmt_cont_name"]).cast("int"),
        )
        .withColumn("F11_BUSN_PROCESS", (~filled(F.col("model_ver_busn_proc_code"))).cast("int"))
        .withColumn("F12_SIGNIFICANCE", (~filled(F.col("significance"))).cast("int"))
        .withColumn(
            "F13_SIGNIFICANCE_MISMATCH",
            (
                filled(F.col("model_ver_signfcnt_ctgry_code"))
                & filled(F.col("anlt_significance_code"))
                & (normalized(F.col("model_ver_signfcnt_ctgry_code")) != normalized(F.col("anlt_significance_code")))
            ).cast("int"),
        )
        .withColumn("F14_DEV_REPORT", (~filled(F.col("model_ver_dev_report_sid"))).cast("int"))
        .withColumn(
            "F15_BITBUCKET_CODE",
            (~(
                has_bitbucket(F.col("model_ver_repstry_link_txt"))
                | has_bitbucket(F.col("model_ver_mlstorage_branch_link_txt"))
            )).cast("int"),
        )
        .withColumn(
            "F16_VALIDATION_RESULT",
            ((F.col("validation_not_required") == 0) & (F.col("has_validation_result") == 0)).cast("int"),
        )
        .withColumn(
            "F17_MONITOR_RESULT",
            (
                filled(F.col("model_ver_montrg_last_sid"))
                & (F.col("monitor_result_available") == 0)
            ).cast("int"),
        )
        .withColumn("F18_MONITOR_RESULT_MISMATCH", F.col("monitor_result_mismatch").cast("int"))
        .withColumn(
            "F19_VALIDATION_DATA",
            (
                (F.col("validation_not_required") == 0)
                & (F.col("has_validation_result") == 1)
                & (F.col("has_validation_data_path") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F20_MONITOR_DATA",
            (
                filled(F.col("model_ver_montrg_last_sid"))
                & (F.col("monitor_result_available") == 1)
                & (F.col("has_monitor_data_path") == 0)
            ).cast("int"),
        )
    )

    flag_columns = [f"F{i:02d}_{name}" for i, name in [
        (1, "MODEL_CODE"), (2, "MODEL_NAME"), (3, "MODEL_CODE_DUP"),
        (4, "VERSION_NUM"), (5, "VERSION_NAME"), (6, "DEV_START"),
        (7, "DEV_END"), (8, "EXPL_START"), (9, "OWNER"),
        (10, "DEVELOPER"), (11, "BUSN_PROCESS"), (12, "SIGNIFICANCE"),
        (13, "SIGNIFICANCE_MISMATCH"), (14, "DEV_REPORT"),
        (15, "BITBUCKET_CODE"), (16, "VALIDATION_RESULT"),
        (17, "MONITOR_RESULT"), (18, "MONITOR_RESULT_MISMATCH"),
        (19, "VALIDATION_DATA"), (20, "MONITOR_DATA"),
    ]]

    base = base.withColumn(
        "issue_count",
        reduce(lambda a, b: a + b, [F.col(c) for c in flag_columns], F.lit(0)),
    )

    # Ровно тот объём информации, который нужен для работы с флагами.
    report_columns = [
        "model_sid",
        "model_code",
        "model_name",
        "model_ver_sid",
        "model_ver_num",
        "model_ver_name",
        "significance",
        "significance_name",
        "model_ver_owner_block_name",
        "model_ver_owner_dprtmt_name",
        "model_ver_owner_cont_name",
        "model_ver_dev_block_name",
        "model_ver_dev_dprtmt_name",
        "model_ver_dev_dprtmt_cont_name",
        "expl_start_dttm",
        "issue_count",
        *flag_columns,
    ]

    report = base.select(*report_columns).orderBy("model_sid", "model_ver_sid")
    return report


## 6. Экспорт в Excel

In [ ]:
def excel_value(value):
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def write_excel(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    wb = Workbook(write_only=True)
    ws = wb.create_sheet("Проверка")
    ws.freeze_panes = "A2"

    headers = list(frame.columns)
    header_cells = []
    for header in headers:
        cell = WriteOnlyCell(ws, value=header)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="24435A")
        header_cells.append(cell)
    ws.append(header_cells)

    for idx in range(1, len(headers) + 1):
        ws.column_dimensions[get_column_letter(idx)].width = 24

    row_count = 0
    for row in frame.toLocalIterator():
        if row_count >= MAX_EXCEL_ROWS:
            raise RuntimeError("Результат превышает лимит одного листа Excel")
        values = []
        for raw in row:
            value = excel_value(raw)
            if isinstance(value, str):
                value = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", value)
                if len(value) > 32_767:
                    value = value[:32_767]
            cell = WriteOnlyCell(ws, value=value)
            if isinstance(value, str):
                cell.data_type = "s"
            values.append(cell)
        ws.append(values)
        row_count += 1

    if headers:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{row_count + 1}"

    temp = path.with_name(path.stem + ".partial.xlsx")
    try:
        wb.save(temp)
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


if __name__ == "__main__":
    run()

## 7. Запуск проверки

In [ ]:
report = build_report(spark, AS_OF).cache()

version_count = report.count()
print("Версий в эксплуатации:", version_count)

report.show(20, truncate=False)

In [ ]:
# Пример точечной проверки версии без повторного расчёта всего пайплайна:
# report.filter(F.col("model_ver_sid") == "123145").show(truncate=False)

## 8. Сохранение результата

In [ ]:
write_excel(report, OUTPUT_FILE)
print("Сохранено:", OUTPUT_FILE.resolve())

In [ ]:
# После завершения анализа:
# report.unpersist()